# The UK carbon budget: a short analysis - solutions

This notebook contains the model solutions for the analysis.

Run the cell below to load the packages and the `carbon_budget` dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

carbon_budget = pd.read_csv("carbon_budget.csv")

## 1. Getting to know the data

Before any analysis, take a look at what you're working with.

- Check what type of object `carbon_budget` is with `type()`.
- Preview the first few rows with `.head()`.
- Check how many rows and columns there are with `.shape`.
- The `len()` function you used on lists also works on a `DataFrame` - it gives the number of rows. Check that it agrees with `.shape`.
- You can pull up the documentation of any function or method with `help()` - try it on `carbon_budget.head`.

In [ ]:
print(type(carbon_budget))
print(carbon_budget.head())
print(carbon_budget.shape)
print(len(carbon_budget))

help(carbon_budget.head)

## 2. Summarising emissions with numpy

Before diving into the full table, let's warm up with a small NumPy array. Under the Balanced Pathway, the projected total emissions (MtCO2e) from surface transport for the years 2025 to 2029 are 99.1, 94.0, 88.5, 82.0 and 75.5.

- Create a NumPy array called `transport_emissions` holding these five values.
- Find the mean of the array with `np.mean()`.
- The values are in millions of tonnes (MtCO2e). Convert them all to thousands of tonnes (ktCO2e) by multiplying the array by `1000`.
- Check the type of the data stored in the array with `.dtype`, and the number of dimensions with `.ndim`.

In [ ]:
transport_emissions = np.array([99.1, 94.0, 88.5, 82.0, 75.5])

print(np.mean(transport_emissions))
print(transport_emissions * 1000)
print(transport_emissions.dtype)
transport_emissions.ndim

## 3. Focusing on a single sector

Let's look at surface transport on its own.

- Set the index of `carbon_budget` to the `sector` column, storing the result as `by_sector`.
- Use `.loc` to pull out all the rows for `"Surface transport"`, storing them as `surface`.
- Find the median and the maximum `total_emissions` for surface transport.

In [ ]:
by_sector = carbon_budget.set_index("sector")
surface = by_sector.loc["Surface transport"]

print(surface["total_emissions"].median())
surface["total_emissions"].max()

## 4. Emissions beyond CO2

`total_emissions` covers all greenhouse gases, while `co2_emissions` is only the carbon-dioxide part. The difference tells us how much comes from other gases, such as methane.

- Add a new column `non_co2_emissions` to `carbon_budget`, equal to `total_emissions` minus `co2_emissions`.
- Preview the result with `.head()`.

In [ ]:
carbon_budget["non_co2_emissions"] = (
    carbon_budget["total_emissions"] - carbon_budget["co2_emissions"]
)

carbon_budget.head()

## 5. Comparing and visualising the scenarios

The dataset has two scenarios: the `"Balanced Pathway"` (the route to net zero) and the `"Baseline"` (no further action).

- Use `.groupby()` to find the total (summed) `total_emissions` for each `scenario`, storing it as `by_scenario`. Which scenario emits more overall?
- List the style sheets you have available with `plt.style.available`, then apply one with `plt.style.use()`.
- Using `set_index()` and `.loc` (as you did for `surface`), extract just the Balanced Pathway rows. Then use `.groupby()` to find the total `total_emissions` for each `sector`, storing it as `by_sector`, and make a bar chart of `by_sector` with `.plot(kind="bar")`, labelling the y-axis. Two sectors have negative totals - can you see why? Save the finished plot with `plt.savefig("emissions_by_sector.png")`.

In [ ]:
by_scenario = carbon_budget.groupby("scenario")["total_emissions"].sum()
print(by_scenario)

print(plt.style.available)

plt.style.use("seaborn-v0_8")

balanced = carbon_budget.set_index("scenario").loc["Balanced Pathway"]
by_sector = balanced.groupby("sector")["total_emissions"].sum()

by_sector.plot(kind="bar")
plt.ylabel("Total emissions 2025-2050 (MtCO2e)")

plt.savefig("emissions_by_sector.png")

## 6. Saving your analysis

You've added a new column, so save your updated dataset so it can be shared.

- Write `carbon_budget` to a CSV file called `"carbon_budget_analysis.csv"` with `.to_csv()` (pass `index=False`).
- Read it back into a `DataFrame` called `reloaded` with `pd.read_csv()`, and preview it with `.head()`.

In [ ]:
carbon_budget.to_csv("carbon_budget_analysis.csv", index=False)

reloaded = pd.read_csv("carbon_budget_analysis.csv")
reloaded.head()

# Bonus questions

In case you finish early! Fergal wrote these ones.

## B1. More on non-co2 emissions

We have our `non_co2_emissions` column from an earlier step. Calculate an additional column which gives the proportion of total emissions which are non-CO2. Call this column `non_co2_proportion`.
After this:
* For each scenario, find the two sectors with the highest and lowest **average** `non_co2_proportion` across the entire period. 
* *Hint 1 : You will find this is easier if you use `.groupby()`*.
* *Hint 2: Make use of `.idxmax()` and `.idxmin()` methods.*
* Write a print statement which summarises your results. Use f-strings (Google this if unsure!).

In [ ]:
reloaded["non_co2_proportion"] = reloaded["non_co2_emissions"] /  reloaded["total_emissions"]

In [ ]:
grouped_non_co2 = reloaded[["sector", "non_co2_proportion"]].groupby("sector").mean()

In [ ]:
print(f"Highest non-co2 proportion is {grouped_non_co2.idxmax().values[0]}, with an average proportion of {grouped_non_co2.max().values[0]} made up by non-CO2 emissions.")
print(f"Lowest non-co2 proportion is {grouped_non_co2.idxmin().values[0]}, with an average proportion of {grouped_non_co2.min().values[0]} made up by non-CO2 emissions.")

## B2. Rates of change

In this question, we will investigate how quickly emissions change in the Balanced Pathway.

* First, filter out the Baseline data, so that you only have Balanced Pathway data.
* To begin, create a new `grouped_by_year` dataframe using the `.groupby()` method.
* Create a new column called `yoy_change_emissions` which uses the `.shift()` method to calculate the YoY change in `total_emissions` from one year to the next.
* Find the year with the largest absolute change in emissions, and the year with the lowest absolute change (excluding NaN values).
* Find the average annual rate of emissions reduction across the entire period.
* Summarise these results in a similar to way to your answer to B1.

In [ ]:
grouped_by_year = reloaded.loc[reloaded["scenario"] == "Balanced Pathway"][["year", "total_emissions"]].groupby("year").sum()

In [ ]:
grouped_by_year["yoy_change_emissions"] = grouped_by_year["total_emissions"] - grouped_by_year["total_emissions"].shift(1)

In [ ]:
grouped_by_year

# B3. Plot the baseline and balanced pathway

* Use `.groupby()` to make a dataframe `pathway_emissions_by_year` which has the total emissions by year, by pathway.
* Use `.pivot()` to get a column for each `scenario`.
* Make a simple lineplot, with one line for each of the baseline and the pathway, and a legend.

In [ ]:
pathway_emissions_by_year = reloaded.groupby(["year", "scenario"]).sum(numeric_only=True).reset_index(level=1)

pathway_emissions_by_year

In [ ]:
final_df = pathway_emissions_by_year[["scenario", "total_emissions"]].pivot(columns="scenario")
final_df.columns = final_df.columns.levels[1]
final_df.columns

In [ ]:
for c in final_df.columns:
    plt.plot(final_df.index, final_df[c])
plt.legend(final_df.columns)